In [0]:
# Cell 1 — Install required libraries
# Runs on: Databricks serverless compute

%pip install litellm pydantic instructor python-dotenv

In [0]:
# Cell 2 — Restart kernel to use newly installed libraries
dbutils.library.restartPython()

In [0]:
# Cell 3 — Verify libraries installed correctly
import pydantic
import instructor
import litellm

print(f"pydantic version: {pydantic.__version__}")
print(f"instructor version: {instructor.__version__}")
print("✅ All libraries loaded successfully")

In [0]:
# Cell 4 — Check available Groq models
import requests

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)

models = response.json()
print("Available Groq models:")
for model in models["data"]:
    print(f"  → {model['id']}")

In [0]:
# Cell 5 — Configure Groq API key
import os

# Paste your Groq API key here
# Do NOT hardcode API keys. Use Databricks secrets instead:
# dbutils.secrets.get(scope="<your-scope>", key="groq_api_key")
os.environ["GROQ_API_KEY"] = dbutils.secrets.get(scope="my_scope", key="groq_api_key")
os.environ["LLM_MODEL"] = "groq/openai/gpt-oss-120b"

print(f"Model set: {os.environ['LLM_MODEL']}")
print("✅ Environment variables set")

In [0]:
# Cell 6 — Test Groq LLM connection
from litellm import completion

response = completion(
    model=os.environ["LLM_MODEL"],
    messages=[
        {
            "role": "user",
            "content": "Say exactly this and nothing else: Groq connection successful"
        }
    ],
    max_tokens=20
)

print(response.choices[0].message.content)
print("✅ Groq connection working")

In [0]:
# Cell 7 — Create sample retail data
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

# ── PRODUCT CATALOG ───────────────────────────────────────
products_data = [
    # Beverages — India
    ("SKU_BEV_001", "Coca-Cola 2L",        "beverages", "India", "INR", 89.0,  45.0,  "Coca-Cola"),
    ("SKU_BEV_002", "Sprite 1L",           "beverages", "India", "INR", 55.0,  28.0,  "Coca-Cola"),
    ("SKU_BEV_003", "Tropicana Orange 1L", "beverages", "India", "INR", 120.0, 65.0,  "PepsiCo"),
    ("SKU_BEV_004", "Red Bull 250ml",      "beverages", "India", "INR", 115.0, 72.0,  "Red Bull"),
    ("SKU_BEV_005", "Minute Maid 1L",      "beverages", "India", "INR", 99.0,  54.0,  "Coca-Cola"),

    # Beverages — USA
    ("SKU_BEV_006", "Gatorade 32oz",       "beverages", "USA",   "USD", 2.99,  1.20,  "PepsiCo"),
    ("SKU_BEV_007", "Powerade 28oz",       "beverages", "USA",   "USD", 1.89,  0.85,  "Coca-Cola"),
    ("SKU_BEV_008", "Snapple Tea 16oz",    "beverages", "USA",   "USD", 2.49,  1.10,  "Snapple"),
    ("SKU_BEV_009", "Arizona Green Tea",   "beverages", "USA",   "USD", 1.29,  0.55,  "Arizona"),
    ("SKU_BEV_010", "Monster Energy 16oz", "beverages", "USA",   "USD", 3.49,  1.80,  "Monster"),

    # Stationery — India
    ("SKU_STA_001", "Classmate Notebook",  "stationery","India", "INR", 45.0,  22.0,  "ITC"),
    ("SKU_STA_002", "Reynolds Pen Pack",   "stationery","India", "INR", 30.0,  12.0,  "Reynolds"),
    ("SKU_STA_003", "Camlin Geometry Box", "stationery","India", "INR", 180.0, 95.0,  "Camlin"),
    ("SKU_STA_004", "Apsara Pencil Pack",  "stationery","India", "INR", 25.0,  10.0,  "Apsara"),
    ("SKU_STA_005", "Stapler Kangaro",     "stationery","India", "INR", 220.0, 110.0, "Kangaro"),

    # Stationery — Germany
    ("SKU_STA_006", "Leitz Binder A4",    "stationery","Germany","EUR", 8.99,  4.20,  "Leitz"),
    ("SKU_STA_007", "Stabilo Highlighter", "stationery","Germany","EUR", 3.49,  1.40,  "Stabilo"),
    ("SKU_STA_008", "Faber Castell Pens",  "stationery","Germany","EUR", 6.99,  3.10,  "Faber"),
    ("SKU_STA_009", "Avery Labels Pack",   "stationery","Germany","EUR", 4.49,  1.90,  "Avery"),
    ("SKU_STA_010", "Herlitz Notebook",    "stationery","Germany","EUR", 2.99,  1.20,  "Herlitz"),

    # Electronics — USA
    ("SKU_ELE_001", "Anker USB-C Cable",   "electronics","USA",  "USD", 15.99, 6.50,  "Anker"),
    ("SKU_ELE_002", "Belkin Phone Stand",  "electronics","USA",  "USD", 24.99, 10.00, "Belkin"),
    ("SKU_ELE_003", "JBL Earbuds",        "electronics","USA",  "USD", 49.99, 22.00, "JBL"),
    ("SKU_ELE_004", "Samsung Fast Charger","electronics","USA",  "USD", 29.99, 13.00, "Samsung"),
    ("SKU_ELE_005", "Logitech Mouse",      "electronics","USA",  "USD", 39.99, 18.00, "Logitech"),

    # Electronics — India
    ("SKU_ELE_006", "boAt Earphones",      "electronics","India","INR", 999.0, 480.0, "boAt"),
    ("SKU_ELE_007", "Syska LED Bulb 9W",   "electronics","India","INR", 199.0, 85.0,  "Syska"),
    ("SKU_ELE_008", "Ambrane Power Bank",  "electronics","India","INR", 1299.0,620.0, "Ambrane"),
    ("SKU_ELE_009", "TP-Link WiFi Adapter","electronics","India","INR", 799.0, 380.0, "TP-Link"),
    ("SKU_ELE_010", "Zebronics Speaker",   "electronics","India","INR", 1499.0,720.0, "Zebronics"),

    # Apparel — USA
    ("SKU_APP_001", "Hanes T-Shirt M",     "apparel",   "USA",  "USD", 12.99, 4.50,  "Hanes"),
    ("SKU_APP_002", "Fruit of Loom Socks", "apparel",   "USA",  "USD", 8.99,  3.20,  "FotL"),
    ("SKU_APP_003", "Gildan Hoodie L",     "apparel",   "USA",  "USD", 29.99, 12.00, "Gildan"),
    ("SKU_APP_004", "Champion Cap",        "apparel",   "USA",  "USD", 19.99, 8.00,  "Champion"),
    ("SKU_APP_005", "Nike Crew Socks",     "apparel",   "USA",  "USD", 14.99, 6.00,  "Nike"),

    # Apparel — India
    ("SKU_APP_006", "Jockey Innerwear M",  "apparel",   "India","INR", 299.0, 140.0, "Jockey"),
    ("SKU_APP_007", "Lux Cozi Vest",       "apparel",   "India","INR", 199.0, 85.0,  "Lux"),
    ("SKU_APP_008", "Dollar Bigboss Trunk", "apparel",  "India","INR", 249.0, 110.0, "Dollar"),
    ("SKU_APP_009", "VIP Frenchie Brief",  "apparel",   "India","INR", 179.0, 75.0,  "VIP"),
    ("SKU_APP_010", "Rupa Frontline Vest", "apparel",   "India","INR", 159.0, 65.0,  "Rupa"),

    # FMCG — India
    ("SKU_FMCG_001","Surf Excel 1kg",      "fmcg",      "India","INR", 230.0, 112.0, "HUL"),
    ("SKU_FMCG_002","Ariel 1kg",           "fmcg",      "India","INR", 245.0, 118.0, "P&G"),
    ("SKU_FMCG_003","Colgate 200g",        "fmcg",      "India","INR", 89.0,  42.0,  "Colgate"),
    ("SKU_FMCG_004","Dove Soap 100g",      "fmcg",      "India","INR", 55.0,  25.0,  "HUL"),
    ("SKU_FMCG_005","Dettol Handwash 200ml","fmcg",     "India","INR", 99.0,  48.0,  "Reckitt"),

    # FMCG — Germany
    ("SKU_FMCG_006","Persil 1.5kg",        "fmcg",      "Germany","EUR",8.99, 4.20,  "Henkel"),
    ("SKU_FMCG_007","Fairy Dish Soap 500ml","fmcg",     "Germany","EUR",3.49, 1.50,  "P&G"),
    ("SKU_FMCG_008","Nivea Cream 150ml",   "fmcg",      "Germany","EUR",4.99, 2.20,  "Nivea"),
    ("SKU_FMCG_009","Oral-B Toothpaste",   "fmcg",      "Germany","EUR",2.99, 1.30,  "P&G"),
    ("SKU_FMCG_010","Domestos Bleach 750ml","fmcg",     "Germany","EUR",2.49, 1.00,  "HUL"),
]

products_schema = StructType([
    StructField("sku_id",        StringType(),  False),
    StructField("product_name",  StringType(),  False),
    StructField("category",      StringType(),  False),
    StructField("geography",     StringType(),  False),
    StructField("currency",      StringType(),  False),
    StructField("base_price",    DoubleType(),  False),
    StructField("cogs",          DoubleType(),  False),
    StructField("brand",         StringType(),  False),
])

products_df = spark.createDataFrame(products_data, products_schema)
products_df.write.mode("overwrite").saveAsTable("products")
print(f"✅ Products table created: {products_df.count()} SKUs")

In [0]:
# Cell 8 — Create inventory table
from pyspark.sql.functions import col

inventory_data = [
    # Beverages — India (some overstocked — good promo candidates)
    ("SKU_BEV_001", 1840, 30, 200),   # overstocked
    ("SKU_BEV_002", 920,  30, 150),   # overstocked
    ("SKU_BEV_003", 450,  30, 180),   # healthy
    ("SKU_BEV_004", 180,  30, 120),   # low stock
    ("SKU_BEV_005", 680,  30, 160),   # overstocked

    # Beverages — USA
    ("SKU_BEV_006", 2400, 30, 300),   # overstocked
    ("SKU_BEV_007", 1100, 30, 250),   # overstocked
    ("SKU_BEV_008", 340,  30, 200),   # healthy
    ("SKU_BEV_009", 890,  30, 280),   # overstocked
    ("SKU_BEV_010", 220,  30, 180),   # healthy

    # Stationery — India
    ("SKU_STA_001", 3200, 30, 400),   # heavily overstocked
    ("SKU_STA_002", 2800, 30, 350),   # heavily overstocked
    ("SKU_STA_003", 480,  30, 120),   # healthy
    ("SKU_STA_004", 1900, 30, 300),   # overstocked
    ("SKU_STA_005", 320,  30, 80),    # healthy

    # Stationery — Germany
    ("SKU_STA_006", 760,  30, 100),   # overstocked
    ("SKU_STA_007", 1200, 30, 150),   # overstocked
    ("SKU_STA_008", 540,  30, 120),   # healthy
    ("SKU_STA_009", 890,  30, 130),   # overstocked
    ("SKU_STA_010", 420,  30, 90),    # healthy

    # Electronics — USA
    ("SKU_ELE_001", 1560, 30, 200),   # overstocked
    ("SKU_ELE_002", 680,  30, 120),   # overstocked
    ("SKU_ELE_003", 290,  30, 80),    # healthy
    ("SKU_ELE_004", 440,  30, 100),   # healthy
    ("SKU_ELE_005", 380,  30, 90),    # healthy

    # Electronics — India
    ("SKU_ELE_006", 920,  30, 150),   # overstocked
    ("SKU_ELE_007", 2100, 30, 280),   # heavily overstocked
    ("SKU_ELE_008", 340,  30, 80),    # healthy
    ("SKU_ELE_009", 480,  30, 90),    # healthy
    ("SKU_ELE_010", 260,  30, 70),    # healthy

    # Apparel — USA
    ("SKU_APP_001", 3400, 30, 400),   # heavily overstocked
    ("SKU_APP_002", 1800, 30, 300),   # overstocked
    ("SKU_APP_003", 680,  30, 150),   # overstocked
    ("SKU_APP_004", 420,  30, 100),   # healthy
    ("SKU_APP_005", 560,  30, 120),   # healthy

    # Apparel — India
    ("SKU_APP_006", 1200, 30, 200),   # overstocked
    ("SKU_APP_007", 980,  30, 180),   # overstocked
    ("SKU_APP_008", 540,  30, 120),   # healthy
    ("SKU_APP_009", 760,  30, 140),   # overstocked
    ("SKU_APP_010", 430,  30, 100),   # healthy

    # FMCG — India
    ("SKU_FMCG_001", 1840, 30, 280),  # overstocked
    ("SKU_FMCG_002", 920,  30, 240),  # overstocked
    ("SKU_FMCG_003", 2400, 30, 350),  # heavily overstocked
    ("SKU_FMCG_004", 1600, 30, 300),  # overstocked
    ("SKU_FMCG_005", 780,  30, 180),  # overstocked

    # FMCG — Germany
    ("SKU_FMCG_006", 640,  30, 120),  # overstocked
    ("SKU_FMCG_007", 1100, 30, 180),  # overstocked
    ("SKU_FMCG_008", 480,  30, 100),  # healthy
    ("SKU_FMCG_009", 920,  30, 150),  # overstocked
    ("SKU_FMCG_010", 560,  30, 110),  # healthy
]

inventory_schema = StructType([
    StructField("sku_id",                  StringType(),  False),
    StructField("current_stock",           IntegerType(), False),
    StructField("target_days_cover",       IntegerType(), False),
    StructField("avg_daily_sales",         IntegerType(), False),
])

inventory_df = spark.createDataFrame(inventory_data, inventory_schema)

# Calculate days of cover
inventory_df = inventory_df.withColumn(
    "days_cover",
    (col("current_stock") / col("avg_daily_sales")).cast("integer")
)

# Flag overstocked items
inventory_df = inventory_df.withColumn(
    "stock_status",
    (
        (col("days_cover") > col("target_days_cover") * 2).cast("string")
    )
)

inventory_df.write.mode("overwrite").saveAsTable("inventory")
print(f"✅ Inventory table created: {inventory_df.count()} SKUs")

# Show overstocked items — best promo candidates
print("\nTop overstocked items (best promotion candidates):")
inventory_df.filter(
    col("days_cover") > col("target_days_cover") * 1.5
).select(
    "sku_id", "current_stock", "days_cover", "target_days_cover"
).orderBy(
    col("days_cover").desc()
).show(10)

In [0]:
# Cell 9 — Verify inventory calculation
from pyspark.sql.functions import col, round as spark_round

inventory_check = spark.sql("""
    SELECT 
        sku_id,
        current_stock,
        avg_daily_sales,
        target_days_cover,
        CAST(current_stock / avg_daily_sales AS INTEGER) as days_cover,
        CASE 
            WHEN CAST(current_stock / avg_daily_sales AS INTEGER) 
                 > target_days_cover * 1.5 
            THEN 'OVERSTOCKED'
            WHEN CAST(current_stock / avg_daily_sales AS INTEGER) 
                 < target_days_cover * 0.5 
            THEN 'LOW STOCK'
            ELSE 'HEALTHY'
        END as stock_status
    FROM inventory
    ORDER BY days_cover DESC
""")

inventory_check.write.mode("overwrite").saveAsTable("inventory_status")

print("✅ Inventory status calculated")
print(f"\nTotal SKUs: {inventory_check.count()}")
print("\nTop 10 overstocked (best promo candidates):")
inventory_check.filter(
    col("stock_status") == "OVERSTOCKED"
).select(
    "sku_id",
    "current_stock", 
    "days_cover",
    "stock_status"
).orderBy(
    col("days_cover").desc()
).show(10)

In [0]:
# Cell 10 — Debug inventory values
spark.sql("""
    SELECT 
        sku_id,
        current_stock,
        avg_daily_sales,
        target_days_cover,
        current_stock / avg_daily_sales as days_cover_raw,
        CAST(current_stock / avg_daily_sales AS INTEGER) as days_cover_int
    FROM inventory
    LIMIT 5
""").show()

In [0]:
# Cell 11 — Recreate inventory with correct daily sales figures
inventory_data_fixed = [
    # Format: sku_id, current_stock, target_days_cover, avg_daily_sales
    # avg_daily_sales should be small (5-30 units/day for most retail)

    # Beverages — India
    ("SKU_BEV_001", 1840, 30, 20),   # 92 days cover — overstocked
    ("SKU_BEV_002", 920,  30, 15),   # 61 days cover — overstocked
    ("SKU_BEV_003", 450,  30, 18),   # 25 days cover — healthy
    ("SKU_BEV_004", 180,  30, 12),   # 15 days cover — low stock
    ("SKU_BEV_005", 680,  30, 16),   # 42 days cover — overstocked

    # Beverages — USA
    ("SKU_BEV_006", 2400, 30, 30),   # 80 days — overstocked
    ("SKU_BEV_007", 1100, 30, 25),   # 44 days — overstocked
    ("SKU_BEV_008", 340,  30, 20),   # 17 days — healthy
    ("SKU_BEV_009", 890,  30, 28),   # 31 days — overstocked
    ("SKU_BEV_010", 220,  30, 18),   # 12 days — low stock

    # Stationery — India
    ("SKU_STA_001", 3200, 30, 40),   # 80 days — heavily overstocked
    ("SKU_STA_002", 2800, 30, 35),   # 80 days — heavily overstocked
    ("SKU_STA_003", 480,  30, 12),   # 40 days — overstocked
    ("SKU_STA_004", 1900, 30, 30),   # 63 days — overstocked
    ("SKU_STA_005", 320,  30, 8),    # 40 days — overstocked

    # Stationery — Germany
    ("SKU_STA_006", 760,  30, 10),   # 76 days — overstocked
    ("SKU_STA_007", 1200, 30, 15),   # 80 days — overstocked
    ("SKU_STA_008", 540,  30, 12),   # 45 days — overstocked
    ("SKU_STA_009", 890,  30, 13),   # 68 days — overstocked
    ("SKU_STA_010", 420,  30, 9),    # 46 days — overstocked

    # Electronics — USA
    ("SKU_ELE_001", 1560, 30, 20),   # 78 days — overstocked
    ("SKU_ELE_002", 680,  30, 12),   # 56 days — overstocked
    ("SKU_ELE_003", 290,  30, 8),    # 36 days — overstocked
    ("SKU_ELE_004", 440,  30, 10),   # 44 days — overstocked
    ("SKU_ELE_005", 380,  30, 9),    # 42 days — overstocked

    # Electronics — India
    ("SKU_ELE_006", 920,  30, 15),   # 61 days — overstocked
    ("SKU_ELE_007", 2100, 30, 28),   # 75 days — heavily overstocked
    ("SKU_ELE_008", 340,  30, 8),    # 42 days — overstocked
    ("SKU_ELE_009", 480,  30, 9),    # 53 days — overstocked
    ("SKU_ELE_010", 260,  30, 7),    # 37 days — overstocked

    # Apparel — USA
    ("SKU_APP_001", 3400, 30, 40),   # 85 days — heavily overstocked
    ("SKU_APP_002", 1800, 30, 30),   # 60 days — overstocked
    ("SKU_APP_003", 680,  30, 15),   # 45 days — overstocked
    ("SKU_APP_004", 420,  30, 10),   # 42 days — overstocked
    ("SKU_APP_005", 560,  30, 12),   # 46 days — overstocked

    # Apparel — India
    ("SKU_APP_006", 1200, 30, 20),   # 60 days — overstocked
    ("SKU_APP_007", 980,  30, 18),   # 54 days — overstocked
    ("SKU_APP_008", 540,  30, 12),   # 45 days — overstocked
    ("SKU_APP_009", 760,  30, 14),   # 54 days — overstocked
    ("SKU_APP_010", 430,  30, 10),   # 43 days — overstocked

    # FMCG — India
    ("SKU_FMCG_001", 1840, 30, 28),  # 65 days — overstocked
    ("SKU_FMCG_002", 920,  30, 24),  # 38 days — overstocked
    ("SKU_FMCG_003", 2400, 30, 35),  # 68 days — heavily overstocked
    ("SKU_FMCG_004", 1600, 30, 30),  # 53 days — overstocked
    ("SKU_FMCG_005", 780,  30, 18),  # 43 days — overstocked

    # FMCG — Germany
    ("SKU_FMCG_006", 640,  30, 12),  # 53 days — overstocked
    ("SKU_FMCG_007", 1100, 30, 18),  # 61 days — overstocked
    ("SKU_FMCG_008", 480,  30, 10),  # 48 days — overstocked
    ("SKU_FMCG_009", 920,  30, 15),  # 61 days — overstocked
    ("SKU_FMCG_010", 560,  30, 11),  # 50 days — overstocked
]

inventory_schema = StructType([
    StructField("sku_id",             StringType(),  False),
    StructField("current_stock",      IntegerType(), False),
    StructField("target_days_cover",  IntegerType(), False),
    StructField("avg_daily_sales",    IntegerType(), False),
])

inventory_df = spark.createDataFrame(
    inventory_data_fixed, inventory_schema
)

inventory_df.write.mode("overwrite").saveAsTable("inventory")

# Recalculate status
inventory_status = spark.sql("""
    SELECT
        sku_id,
        current_stock,
        avg_daily_sales,
        target_days_cover,
        CAST(current_stock / avg_daily_sales AS INTEGER) as days_cover,
        CASE
            WHEN CAST(current_stock / avg_daily_sales AS INTEGER)
                 > target_days_cover * 1.5
            THEN 'OVERSTOCKED'
            WHEN CAST(current_stock / avg_daily_sales AS INTEGER)
                 < target_days_cover * 0.5
            THEN 'LOW_STOCK'
            ELSE 'HEALTHY'
        END as stock_status
    FROM inventory
    ORDER BY days_cover DESC
""")

inventory_status.write.mode("overwrite").saveAsTable("inventory_status")

print("✅ Inventory recreated with correct values")
print(f"Total SKUs: {inventory_status.count()}")
print("\nTop 10 overstocked (best promo candidates):")
inventory_status.filter(
    "stock_status = 'OVERSTOCKED'"
).select(
    "sku_id", "current_stock",
    "days_cover", "stock_status"
).orderBy("days_cover", ascending=False).show(10)

In [0]:
# Cell 12 — Create competitor prices table
competitor_data = [
    # sku_id, competitor_name, competitor_price, currency, geography

    # Beverages — India
    ("SKU_BEV_001", "DMart",       85.0,  "INR", "India"),
    ("SKU_BEV_001", "BigBazaar",   92.0,  "INR", "India"),
    ("SKU_BEV_002", "DMart",       52.0,  "INR", "India"),
    ("SKU_BEV_002", "BigBazaar",   58.0,  "INR", "India"),
    ("SKU_BEV_003", "DMart",       115.0, "INR", "India"),
    ("SKU_BEV_003", "BigBazaar",   125.0, "INR", "India"),
    ("SKU_BEV_005", "DMart",       95.0,  "INR", "India"),
    ("SKU_BEV_005", "BigBazaar",   102.0, "INR", "India"),

    # Beverages — USA
    ("SKU_BEV_006", "Walmart",     2.79,  "USD", "USA"),
    ("SKU_BEV_006", "Target",      3.09,  "USD", "USA"),
    ("SKU_BEV_007", "Walmart",     1.79,  "USD", "USA"),
    ("SKU_BEV_007", "Target",      1.99,  "USD", "USA"),
    ("SKU_BEV_009", "Walmart",     1.19,  "USD", "USA"),
    ("SKU_BEV_010", "Walmart",     3.29,  "USD", "USA"),

    # Stationery — India
    ("SKU_STA_001", "DMart",       42.0,  "INR", "India"),
    ("SKU_STA_001", "BigBazaar",   48.0,  "INR", "India"),
    ("SKU_STA_002", "DMart",       28.0,  "INR", "India"),
    ("SKU_STA_004", "DMart",       22.0,  "INR", "India"),
    ("SKU_STA_004", "BigBazaar",   27.0,  "INR", "India"),

    # Stationery — Germany
    ("SKU_STA_006", "Kaufland",    8.49,  "EUR", "Germany"),
    ("SKU_STA_007", "Kaufland",    3.29,  "EUR", "Germany"),
    ("SKU_STA_007", "Rewe",        3.59,  "EUR", "Germany"),
    ("SKU_STA_008", "Kaufland",    6.49,  "EUR", "Germany"),

    # Electronics — USA
    ("SKU_ELE_001", "Amazon",      13.99, "USD", "USA"),
    ("SKU_ELE_001", "BestBuy",     16.99, "USD", "USA"),
    ("SKU_ELE_002", "Amazon",      22.99, "USD", "USA"),
    ("SKU_ELE_003", "Amazon",      44.99, "USD", "USA"),
    ("SKU_ELE_003", "BestBuy",     52.99, "USD", "USA"),

    # Electronics — India
    ("SKU_ELE_006", "Amazon",      949.0, "INR", "India"),
    ("SKU_ELE_006", "Flipkart",    929.0, "INR", "India"),
    ("SKU_ELE_007", "Amazon",      189.0, "INR", "India"),
    ("SKU_ELE_007", "Flipkart",    179.0, "INR", "India"),

    # Apparel — USA
    ("SKU_APP_001", "Walmart",     11.99, "USD", "USA"),
    ("SKU_APP_001", "Target",      13.49, "USD", "USA"),
    ("SKU_APP_002", "Walmart",     7.99,  "USD", "USA"),
    ("SKU_APP_003", "Walmart",     27.99, "USD", "USA"),

    # Apparel — India
    ("SKU_APP_006", "DMart",       279.0, "INR", "India"),
    ("SKU_APP_006", "Amazon",      269.0, "INR", "India"),
    ("SKU_APP_007", "DMart",       189.0, "INR", "India"),

    # FMCG — India
    ("SKU_FMCG_001", "DMart",      219.0, "INR", "India"),
    ("SKU_FMCG_001", "BigBazaar",  225.0, "INR", "India"),
    ("SKU_FMCG_002", "DMart",      235.0, "INR", "India"),
    ("SKU_FMCG_003", "DMart",      82.0,  "INR", "India"),
    ("SKU_FMCG_004", "DMart",      49.0,  "INR", "India"),
    ("SKU_FMCG_005", "DMart",      92.0,  "INR", "India"),

    # FMCG — Germany
    ("SKU_FMCG_006", "Aldi",       7.99,  "EUR", "Germany"),
    ("SKU_FMCG_006", "Lidl",       8.49,  "EUR", "Germany"),
    ("SKU_FMCG_007", "Aldi",       2.99,  "EUR", "Germany"),
    ("SKU_FMCG_008", "Aldi",       4.49,  "EUR", "Germany"),
    ("SKU_FMCG_009", "Aldi",       2.49,  "EUR", "Germany"),
]

competitor_schema = StructType([
    StructField("sku_id",           StringType(), False),
    StructField("competitor_name",  StringType(), False),
    StructField("competitor_price", DoubleType(), False),
    StructField("currency",         StringType(), False),
    StructField("geography",        StringType(), False),
])

competitor_df = spark.createDataFrame(
    competitor_data, competitor_schema
)
competitor_df.write.mode("overwrite").saveAsTable("competitor_prices")

print(f"✅ Competitor prices: {competitor_df.count()} records")
print("\nSample:")
competitor_df.show(5)

In [0]:
# Cell 12a — Install libraries and load configuration
# Runs on: Databricks serverless compute

%pip install litellm pydantic instructor python-dotenv

In [0]:
# Cell 12a — Load libraries and configuration (safe to commit)
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import col
from litellm import completion

# Load API key from Databricks Secrets (never hardcoded)
os.environ["GROQ_API_KEY"] = dbutils.secrets.get(
    scope="my_scope",
    key="groq_api_key"
)
os.environ["LLM_MODEL"] = "groq/openai/gpt-oss-120b"

# Spark session
spark = SparkSession.builder.getOrCreate()

print("✅ All imports loaded")
print(f"✅ Model: {os.environ['LLM_MODEL']}")
print("✅ API key loaded from Databricks Secrets")

In [0]:
# Cell 13 — Create customer segments table
segments_data = [
    # segment_name, geography, category, size,
    # price_sensitivity, avg_basket, discount_response,
    # loyalty_score, description

    # India segments
    ("premium",        "India", "all",       8000,  0.2, 2500.0, 1.2, 0.85,
     "High income, brand conscious, low price sensitivity"),
    ("mid_income",     "India", "all",       24000, 0.6, 850.0,  2.4, 0.65,
     "Middle income, value seeking, responds to bundles"),
    ("price_sensitive","India", "all",       31000, 0.9, 340.0,  3.2, 0.40,
     "Budget conscious, high discount response"),
    ("loyalty_member", "India", "all",       12000, 0.4, 1200.0, 1.8, 0.92,
     "Loyal customers, moderate price sensitivity"),
    ("lapsed",         "India", "all",       9000,  0.7, 680.0,  2.8, 0.20,
     "Previously active, not purchased in 60+ days"),

    # USA segments
    ("premium",        "USA",   "all",       15000, 0.2, 180.0,  1.1, 0.80,
     "High income, brand conscious"),
    ("mid_income",     "USA",   "all",       42000, 0.5, 85.0,   2.1, 0.60,
     "Middle income, value seeking"),
    ("price_sensitive","USA",   "all",       38000, 0.8, 45.0,   3.0, 0.35,
     "Budget conscious, coupon users"),
    ("loyalty_member", "USA",   "all",       18000, 0.3, 120.0,  1.6, 0.90,
     "Rewards members, high retention"),
    ("lapsed",         "USA",   "all",       11000, 0.6, 65.0,   2.5, 0.18,
     "Inactive 90+ days, win-back needed"),

    # Germany segments
    ("premium",        "Germany","all",      6000,  0.2, 95.0,   1.1, 0.82,
     "Quality focused, low price sensitivity"),
    ("mid_income",     "Germany","all",      18000, 0.5, 55.0,   2.0, 0.62,
     "Value conscious, responds to quality deals"),
    ("price_sensitive","Germany","all",      22000, 0.8, 32.0,   2.8, 0.38,
     "Aldi/Lidl shoppers, very price conscious"),
    ("loyalty_member", "Germany","all",      8000,  0.3, 72.0,   1.5, 0.88,
     "Club card members, habitual buyers"),
    ("lapsed",         "Germany","all",      5000,  0.6, 48.0,   2.3, 0.15,
     "Inactive 60+ days"),
]

segments_schema = StructType([
    StructField("segment_name",       StringType(), False),
    StructField("geography",          StringType(), False),
    StructField("category",           StringType(), False),
    StructField("segment_size",       IntegerType(),False),
    StructField("price_sensitivity",  DoubleType(), False),
    StructField("avg_basket_value",   DoubleType(), False),
    StructField("discount_response",  DoubleType(), False),
    StructField("loyalty_score",      DoubleType(), False),
    StructField("description",        StringType(), False),
])

segments_df = spark.createDataFrame(
    segments_data, segments_schema
)
segments_df.write.mode("overwrite").saveAsTable("customer_segments")

print(f"✅ Customer segments: {segments_df.count()} records")
print("\nSegments by geography:")
segments_df.groupBy("geography").count().show()

In [0]:
# Cell 14 — Holiday calendar 2026-2027 (verified dates)
holidays_data = [
    # INDIA — verified from public sources
    ("Navratri",         "India", "2026-10-11", "2026-10-20",
     "apparel",     1.6, "9 nights of traditional wear demand"),
    ("Dussehra",         "India", "2026-10-20", "2026-10-20",
     "apparel",     1.8, "Victory of good over evil — apparel spike"),
    ("Diwali",           "India", "2026-11-06", "2026-11-10",
     "all",         2.8, "Festival of lights — peak retail 5-day window"),
    ("Christmas",        "India", "2026-12-25", "2026-12-25",
     "all",         1.4, "Christmas shopping — urban India"),
    ("New Year Sale",    "India", "2026-12-26", "2027-01-05",
     "apparel",     2.0, "End of year clearance season"),
    ("Republic Day",     "India", "2027-01-26", "2027-01-26",
     "all",         1.3, "National holiday sale"),
    ("Holi",             "India", "2027-03-22", "2027-03-22",
     "fmcg",        2.1, "Festival of colors — FMCG and beverages spike"),
    ("IPL Season",       "India", "2027-03-20", "2027-05-30",
     "beverages",   1.9, "Cricket season — beverages and snacks spike"),
    ("Back to School",   "India", "2027-06-01", "2027-06-30",
     "stationery",  3.2, "Stationery peak — school reopening season"),
    ("Independence Day", "India", "2027-08-15", "2027-08-15",
     "all",         1.3, "National holiday — patriotic retail"),

    # USA — verified from public sources
    ("Halloween",        "USA",   "2026-10-31", "2026-10-31",
     "fmcg",        1.8, "Candy and costume demand spike"),
    ("Thanksgiving",     "USA",   "2026-11-26", "2026-11-26",
     "fmcg",        2.2, "Grocery and FMCG peak — family gatherings"),
    ("Black Friday",     "USA",   "2026-11-27", "2026-11-27",
     "electronics", 4.2, "Biggest US shopping day — electronics peak"),
    ("Cyber Monday",     "USA",   "2026-11-30", "2026-11-30",
     "electronics", 3.8, "Online electronics and deals peak"),
    ("Christmas",        "USA",   "2026-12-20", "2026-12-25",
     "all",         3.1, "Peak gifting season — all categories"),
    ("New Year",         "USA",   "2026-12-31", "2027-01-01",
     "beverages",   1.9, "New Year celebrations — beverages spike"),
    ("Super Bowl LXI",   "USA",   "2027-02-07", "2027-02-07",
     "beverages",   2.4, "Biggest beverages and snacks day of year"),
    ("Easter",           "USA",   "2027-03-28", "2027-03-28",
     "fmcg",        1.7, "Spring FMCG and grocery spike"),
    ("Back to School",   "USA",   "2027-08-01", "2027-09-05",
     "stationery",  2.9, "Stationery and apparel back to school"),
    ("Black Friday",     "USA",   "2027-11-26", "2027-11-26",
     "electronics", 4.0, "Black Friday 2027 — electronics"),

    # GERMANY — verified from official Munich sources
    ("Oktoberfest",      "Germany","2026-09-19","2026-10-04",
     "beverages",   2.6, "World largest beer festival — Munich (official)"),
    ("Black Friday",     "Germany","2026-11-27","2026-11-27",
     "electronics", 3.2, "Black Friday Germany — electronics deals"),
    ("Christmas",        "Germany","2026-12-20","2026-12-25",
     "all",         3.0, "Peak gifting season Germany"),
    ("New Year Sale",    "Germany","2026-12-26","2027-01-05",
     "all",         2.1, "Post Christmas clearance sales"),
    ("Easter",           "Germany","2027-04-05","2027-04-06",
     "fmcg",        1.9, "Spring FMCG and grocery Germany"),
    ("Summer Sale",      "Germany","2027-07-01","2027-07-31",
     "apparel",     2.3, "Summer apparel clearance Germany"),
    ("Back to School",   "Germany","2027-08-15","2027-09-15",
     "stationery",  2.7, "Stationery peak Germany"),
    ("Oktoberfest 2027", "Germany","2027-09-18","2027-10-03",
     "beverages",   2.6, "Oktoberfest 2027 Munich"),
]

holidays_schema = StructType([
    StructField("event_name",         StringType(), False),
    StructField("geography",          StringType(), False),
    StructField("start_date",         StringType(), False),
    StructField("end_date",           StringType(), False),
    StructField("category_relevance", StringType(), False),
    StructField("demand_multiplier",  DoubleType(), False),
    StructField("description",        StringType(), False),
])

holidays_df = spark.createDataFrame(
    holidays_data, holidays_schema
)
holidays_df.write.mode("overwrite").saveAsTable("holiday_calendar")

print(f"✅ Holiday calendar: {holidays_df.count()} events")
print("\nEvents by geography:")
holidays_df.groupBy("geography").count().show()
print("\nUpcoming high-impact events:")
holidays_df.select(
    "event_name", "geography",
    "start_date", "demand_multiplier"
).orderBy("start_date").show(15)

In [0]:
# Cell 15 — Benchmark elasticity table (cold start data)
# Source: Retail industry research benchmarks
benchmarks_data = [
    # category, geography, price_elasticity,
    # cannibalization_rate, seasonal_uplift_base,
    # typical_discount_response, source_note

    # Beverages
    ("beverages", "India",   2.1, 0.18, 1.4,
     "high",   "FMCG India retail benchmarks"),
    ("beverages", "USA",     1.8, 0.15, 1.3,
     "high",   "US CPG industry benchmarks"),
    ("beverages", "Germany", 1.6, 0.14, 1.2,
     "medium", "European FMCG benchmarks"),
    ("beverages", "global",  1.8, 0.16, 1.3,
     "high",   "Global retail benchmarks"),

    # Stationery
    ("stationery", "India",   2.4, 0.22, 2.1,
     "high",   "Indian education retail benchmarks"),
    ("stationery", "USA",     2.0, 0.18, 1.9,
     "high",   "US office supplies benchmarks"),
    ("stationery", "Germany", 1.8, 0.16, 1.8,
     "medium", "European stationery benchmarks"),
    ("stationery", "global",  2.1, 0.19, 1.9,
     "high",   "Global stationery benchmarks"),

    # Electronics
    ("electronics", "India",   1.4, 0.28, 1.2,
     "medium", "Indian consumer electronics benchmarks"),
    ("electronics", "USA",     1.2, 0.25, 1.8,
     "medium", "US electronics retail benchmarks"),
    ("electronics", "Germany", 1.3, 0.24, 1.4,
     "medium", "European electronics benchmarks"),
    ("electronics", "global",  1.3, 0.26, 1.5,
     "medium", "Global electronics benchmarks"),

    # Apparel
    ("apparel", "India",   2.2, 0.20, 1.8,
     "high",   "Indian fashion retail benchmarks"),
    ("apparel", "USA",     2.0, 0.18, 1.6,
     "high",   "US apparel retail benchmarks"),
    ("apparel", "Germany", 1.8, 0.16, 1.5,
     "medium", "European fashion benchmarks"),
    ("apparel", "global",  2.0, 0.18, 1.6,
     "high",   "Global apparel benchmarks"),

    # FMCG
    ("fmcg", "India",   2.3, 0.24, 1.5,
     "high",   "Indian FMCG benchmarks"),
    ("fmcg", "USA",     1.9, 0.20, 1.3,
     "high",   "US FMCG benchmarks"),
    ("fmcg", "Germany", 1.7, 0.18, 1.2,
     "medium", "European FMCG benchmarks"),
    ("fmcg", "global",  2.0, 0.21, 1.3,
     "high",   "Global FMCG benchmarks"),
]

benchmarks_schema = StructType([
    StructField("category",             StringType(), False),
    StructField("geography",            StringType(), False),
    StructField("price_elasticity",     DoubleType(), False),
    StructField("cannibalization_rate", DoubleType(), False),
    StructField("seasonal_uplift_base", DoubleType(), False),
    StructField("discount_response",    StringType(), False),
    StructField("source_note",          StringType(), False),
])

benchmarks_df = spark.createDataFrame(
    benchmarks_data, benchmarks_schema
)
benchmarks_df.write.mode("overwrite").saveAsTable(
    "benchmark_elasticity"
)

print(f"✅ Benchmarks: {benchmarks_df.count()} records")
print("\nElasticity by category (India):")
benchmarks_df.filter(
    "geography = 'India'"
).select(
    "category",
    "price_elasticity",
    "cannibalization_rate",
    "discount_response"
).show()

In [0]:
# Cell 16 — Verify all Delta Lake tables created
print("=" * 50)
print("DELTA LAKE TABLE SUMMARY")
print("=" * 50)

tables = [
    ("products",            "Product catalog"),
    ("inventory",           "Raw inventory"),
    ("inventory_status",    "Inventory with status"),
    ("competitor_prices",   "Competitor pricing"),
    ("customer_segments",   "Customer segments"),
    ("holiday_calendar",    "Holiday calendar"),
    ("benchmark_elasticity","Elasticity benchmarks"),
]

total_records = 0
all_good = True

for table_name, description in tables:
    try:
        count = spark.sql(
            f"SELECT COUNT(*) as cnt FROM {table_name}"
        ).collect()[0]["cnt"]
        total_records += count
        print(f"✅ {table_name:<25} {count:>4} records  — {description}")
    except Exception as e:
        print(f"❌ {table_name:<25} MISSING — {description}")
        all_good = False

print("=" * 50)
print(f"Total records across all tables: {total_records}")

if all_good:
    print("\n✅ ALL TABLES READY — Hour 1 complete")
    print("   Data foundation is solid for AI engine")
else:
    print("\n❌ Some tables missing — check above")

In [0]:
# Cell 17 — Save notebook checkpoint
print("Notebook: 01_data_setup")
print("Status: Complete")
print("Tables created: 7")
print("Records loaded: 263")
print("Next: 02_ai_engine notebook")
print("\nCommit this notebook to GitHub before Hour 2.")

In [0]:
# Verify secret retrieval works
import os

os.environ["GROQ_API_KEY"] = dbutils.secrets.get(
    scope="my_scope",
    key="groq_api_key"
)
os.environ["LLM_MODEL"] = "groq/openai/gpt-oss-120b"

# Confirm key loaded (shows [REDACTED] for security)
print(f"GROQ_API_KEY: {dbutils.secrets.get(scope='my_scope', key='groq_api_key')}")
print(f"Model: {os.environ['LLM_MODEL']}")
print("✅ Secret loaded successfully")